# Chat With Your Documents
## Notebook 6 — AI Engineer Practical Series

Building a conversational RAG chatbot over a real PDF.
This is the most common enterprise AI use case — query your own documents.

### What we cover:
1. Load a real PDF
2. Chunk & index into Chroma
3. Add conversation memory
4. Answer follow-up questions with context
5. Full conversational RAG pipeline

In [1]:
# Cell 1 — Install dependencies
# Run once, then restart kernel

%pip install langchain langchain-groq langchain-huggingface
%pip install langchain_community langchain-text-splitters langchain_core
%pip install chromadb sentence-transformers pypdf
%pip install python-dotenv numpy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2 — Setup

import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.embeddings import FastEmbedEmbeddings

load_dotenv()

llm = ChatGroq(
    api_key=os.getenv('GROQ_API_KEY'),
    model_name=os.getenv('GROQ_MODEL')
)

print('LLM ready ✅')

'''embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2'
)'''

embeddings = FastEmbedEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"  # Default model, good for RAG
)

print('Embeddings ready ✅')

C:\Users\adars\AppData\Local\Temp\ipykernel_15452\2976471856.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import FastEmbedEmbeddings


LLM ready ✅
Embeddings ready ✅


## Step 1: Load a PDF
Using LangChain's PyPDFLoader to load any PDF as documents.
Each page becomes a separate Document object.

We're loading **Attention Is All You Need** — the original transformer paper.
You can replace `pdf_path` with any local PDF.

In [3]:
# Cell 4 — Load PDF

from langchain_community.document_loaders import PyPDFLoader
import urllib.request

# Download Attention Is All You Need paper
pdf_url = 'https://arxiv.org/pdf/1706.03762'
pdf_path = 'attention_paper.pdf'

print('Downloading PDF...')
urllib.request.urlretrieve(pdf_url, pdf_path)
print(f'Downloaded ✅ — {pdf_path}')

# Load
loader = PyPDFLoader(pdf_path)
pages = loader.load()

print(f'Pages loaded: {len(pages)}')
print(f'\nSample page content:\n{pages[0].page_content[:300]}...')


Downloaded ✅ — attention_paper.pdf
Pages loaded: 15

Sample page content:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par...


## Step 2: Chunk & Index
Split the PDF into overlapping chunks and store in Chroma vector store.

**Why overlap?** A chunk boundary might split a sentence mid-way.
Overlap ensures no context is lost at chunk edges.

In [4]:
# Cell 6 — Chunk & Index

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

import shutil
if os.path.exists('./chroma_attention'):
    shutil.rmtree('./chroma_attention')

# Chunk
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)
chunks = splitter.split_documents(pages)
print(f'Total chunks: {len(chunks)}')

# Index into Chroma
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='./chroma_attention'
)

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

print(f'Indexed ✅ — {vectorstore._collection.count()} chunks in vector store')

Total chunks: 164
Indexed ✅ — 164 chunks in vector store


## Step 3: Conversation Memory
Standard RAG has no memory — each question is independent.
We add memory so the model can answer follow-up questions.

### How it works:
- Store chat history as a list of messages
- Inject history into every prompt via `MessagesPlaceholder`
- Model sees full conversation context on every call

In [5]:
# Cell 8 — Conversational RAG chain with question extraction

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

prompt = ChatPromptTemplate.from_messages([
    ('system', """You are an expert assistant for answering questions about documents.
Answer using ONLY the provided context.
If the answer is not in the context say 'Not found in document.'
Be concise and precise.

Context: {context}"""),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human', '{question}')
])

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# Extract just the question string before passing to retriever
chain = (
    {
        'context': RunnableLambda(lambda x: x['question']) | retriever | format_docs,
        'question': RunnableLambda(lambda x: x['question']),
        'chat_history': RunnableLambda(lambda x: x['chat_history'])
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("Conversational RAG chain ready ✅")

Conversational RAG chain ready ✅


## Step 4: Chat Function
Manages conversation history automatically.
Every call appends to `chat_history` so the model remembers previous turns.

In [6]:
# Cell 10 — Chat function

chat_history = []

def chat(question):
    response = chain.invoke({
        'question': question,
        'chat_history': chat_history
    })
    
    # Update history
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))
    
    print(f'You: {question}')
    print(f'Bot: {response}')
    print(f'History length: {len(chat_history)} messages')
    print('─' * 50)
    
    return response

print('Chat function ready ✅')

Chat function ready ✅


## Step 5: Full Conversation Test
Testing multi-turn conversation.
The last question — 'summarise what you told me' — tests memory directly.
If it summarises correctly, memory is working.

In [9]:
# Cell 12 — Test conversation

# Reset for clean test
chat_history = []

chat('What is this paper/document about?')
chat('Who are the authors?')
chat('How many attention heads does the model use?')
chat('What problem does the attention mechanism solve?')
chat('Can you summarise what you told me so far?')  # Tests memory

You: What is this paper/document about?
Bot: "Attention Is All You Need".
History length: 2 messages
──────────────────────────────────────────────────
You: Who are the authors?
Bot: The authors are:
1. Ashish Vaswani
2. Noam Shazeer
3. Niki Parmar
4. Jakob Uszkoreit
5. Llion Jones
6. Aidan N. Gomez
7. Łukasz Kaiser 
Note that the list of authors appears twice in the document.
History length: 4 messages
──────────────────────────────────────────────────
You: How many attention heads does the model use?
Bot: The model employs h = 8 parallel attention layers, or heads.
History length: 6 messages
──────────────────────────────────────────────────
You: What problem does the attention mechanism solve?
Bot: The attention mechanism allows modeling of dependencies without regard to their distance in the input sequence.
History length: 8 messages
──────────────────────────────────────────────────
You: Can you summarise what you told me so far?
Bot: This document appears to be a study about the 

'This document appears to be a study about the "Attention Is All You Need" paper, which uses an attention mechanism to process sequences. Specifically, it discusses a model with 8 attention heads, allowing it to model dependencies without regard to distance in the input sequence.'

## Step 6: Debug — Inspect Retrieved Chunks
See exactly which chunks are retrieved for each question.
This is what LangSmith does in production — tracing every retrieval step.

Good for diagnosing bad answers:
- Wrong answer → wrong chunks retrieved → fix chunking or retrieval strategy
- Right chunks, wrong answer → fix the prompt

In [10]:
# Cell 14 — Debug retrieval

def chat_debug(question):
    # Show retrieved chunks
    docs = retriever.invoke(question)
    print(f'Question: {question}')
    print(f'\nRetrieved {len(docs)} chunks:')
    for i, doc in enumerate(docs):
        print(f'\n[{i}] Page {doc.metadata.get("page", "?")}: {doc.page_content[:200]}...')
    
    # Get answer
    response = chain.invoke({
        'question': question,
        'chat_history': chat_history
    })
    print(f'\nAnswer: {response}')
    print('─' * 50)

chat_debug('What is multi-head attention?')
chat_debug('What are the encoder and decoder components?')

Question: What is multi-head attention?

Retrieved 3 chunks:

[0] Page 3: Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
of the values, ...

[1] Page 4: is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-deco...

[2] Page 1: to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism rel...

Answer: Multi-Head Attention consists of several attention layers running in parallel.
──────────────────────────────────────────────────
Question: What are the encoder and decoder components?

Retrieved 3 chunks:

[0] Page 2: respectively.
3.1 Encoder and Decoder Stacks
Encoder: The en

## Summary

### What we built:
- PDF loader → chunks → Chroma vector store
- Conversational RAG with persistent memory
- Debug view showing retrieved chunks per question

### Key concepts:

| Concept | What it does |
|---|---|
| `PyPDFLoader` | Loads PDF pages as Document objects |
| `RecursiveCharacterTextSplitter` | Splits docs into overlapping chunks |
| `Chroma` | Local vector store — indexes and retrieves chunks |
| `MessagesPlaceholder` | Injects chat history into the prompt |
| `HumanMessage / AIMessage` | LangChain message types for history |
| `chat_debug()` | Shows which chunks were retrieved — production debugging pattern |

### Diagnosis framework:
- **Wrong answer + wrong chunks** → fix chunking strategy or retrieval (k, chunk_size)
- **Right chunks + wrong answer** → fix the prompt
- **No chunks** → fix embeddings or query

### Next: LangChain Agents
LLMs that decide which tools to use and when — the foundation of agentic AI systems.